In [2]:
from dataclasses import dataclass

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator

@dataclass
class AdderResources:
    """Simple container for resource estimates of an n-bit adder.

    This is a first-order, approximate model corresponding to the
    naive ripple-carry adder implemented below.
    """
    n_bits: int
    toffoli_count: int
    cnot_count: int
    ancilla_count: int

In [3]:
def inplace_ripple_adder(circ: QuantumCircuit, x, y, c):
    """
    In-place ripple-carry adder.

    Logical effect (assuming c[0] = 0 and c[1..n] = 0):
        |x⟩ |y⟩ |0...0⟩  ↦  |x⟩ |x + y (mod 2^n)⟩ |0...0⟩

    Args:
        circ: QuantumCircuit to which the adder gates are added.
        x:    sequence of n qubits for the first addend (LSB at index 0).
        y:    sequence of n qubits for the second addend; will be overwritten with the sum.
        c:    sequence of n+1 qubits for carries; c[0] is the global carry-in.
              All c[i] are assumed to start in |0⟩ and are returned to |0⟩.
    """
    n = len(x)
    assert len(y) == n, "x and y must have the same length"
    assert len(c) == n + 1, "carry register must have length n+1"

    # ---------- Forward pass: compute carries into c[1..n] ----------
    # For each bit i, compute:
    #   c[i+1] ^= (x[i] & y[i]) ⊕ (x[i] & c[i]) ⊕ (y[i] & c[i])
    for i in range(n):
        circ.ccx(x[i], y[i], c[i + 1])
        circ.ccx(x[i], c[i], c[i + 1])
        circ.ccx(y[i], c[i], c[i + 1])

    # ---------- Backward pass: uncompute carries, write sum into y ----------
    # Process bits from MSB down to LSB.
    for i in reversed(range(n)):
        # Uncompute c[i+1] (reverse the forward-pass carry logic).
        circ.ccx(y[i], c[i], c[i + 1])
        circ.ccx(x[i], c[i], c[i + 1])
        circ.ccx(x[i], y[i], c[i + 1])

        # Now write the sum bit into y[i]:
        #   y[i] ^= x[i]
        #   y[i] ^= c[i]
        circ.cx(x[i], y[i])
        circ.cx(c[i], y[i])

In [4]:
def inplace_ripple_adder_circuit(n: int) -> QuantumCircuit:
    """
    Build an n-bit in-place ripple-carry adder circuit.

    Registers:
        x (n): input addend, preserved
        y (n): input addend, overwritten with x + y (mod 2^n)
        c (n+1): carry ancillas, all returned to |0⟩

    Initial state should be:
        |x⟩_x |y⟩_y |0...0⟩_c
    Final state:
        |x⟩_x |x + y (mod 2^n)⟩_y |0...0⟩_c
    """
    x = QuantumRegister(n, "x")
    y = QuantumRegister(n, "y")
    c = QuantumRegister(n + 1, "c")  # c[0] is cin, c[1..n] are carries

    circ = QuantumCircuit(x, y, c)
    inplace_ripple_adder(circ, x, y, c)
    return circ

In [5]:
from qiskit import ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit import transpile

sim = AerSimulator()


def prepare_inputs(base_circ: QuantumCircuit, x_val: int, y_val: int, n: int) -> QuantumCircuit:
    """Return a circuit where |x_val> and |y_val> are prepared before running the adder."""
    prep = QuantumCircuit(*base_circ.qregs)

    x_reg = prep.qregs[0]  # x
    y_reg = prep.qregs[1]  # y

    # Set bits of x
    for i in range(n):
        if (x_val >> i) & 1:
            prep.x(x_reg[i])

    # Set bits of y
    for i in range(n):
        if (y_val >> i) & 1:
            prep.x(y_reg[i])

    # Run the ripple-carry adder after the inputs are prepared
    composed_circ = prep.compose(base_circ)
    # compose may be typed to return Optional[QuantumCircuit]; ensure it's not None for the declared return type
    assert composed_circ is not None
    return composed_circ


def run_case_sum_only(x_val: int, y_val: int, n: int = 2):
    base = inplace_ripple_adder_circuit(n)
    qc = prepare_inputs(base, x_val, y_val, n)

    # Classical register layout: first n bits store the sum, the final bit stores cout
    c = ClassicalRegister(n + 1, "c")
    qc.add_register(c)

    s_reg = qc.qregs[4]
    cout_reg = qc.qregs[3]

    # Measure the sum (bit i -> classical bit i)
    for i in range(n):
        qc.measure(s_reg[i], c[i])

    # Measure the final carry-out (most-significant carry) into the last classical bit
    qc.measure(cout_reg[n - 1], c[n])

    tqc = transpile(qc, sim)
    result = sim.run(tqc, shots=1024).result()
    counts = result.get_counts()

    # There should be a single outcome; get its bitstring (drop register separators)
    bitstring = max(counts, key=counts.get).replace(" ", "")

    carry_bit = bitstring[0] if bitstring else "0"
    sum_bits = bitstring[1:] if len(bitstring) > 1 else ""

    sum_val = int(sum_bits, 2) if sum_bits else 0
    cout_val = int(carry_bit, 2)

    print(
        f"x = {x_val}, y = {y_val}  →  measured sum s = {sum_val} (bits {sum_bits}), "
        f"cout (final carry) = {cout_val} (bit {carry_bit})"
    )

In [6]:
for x in range(4):
    for y in range(4):
        run_case_sum_only(x, y, n=2)

CircuitError: 'register name "c" already exists'